# Gen AI: Dynamic Executive Summary
Synthesizes gold aggregates and pre-computed statistical summaries into a structured, 3-bullet executive briefing using serverless Foundation Model APIs (`ai_query`) with strict factual grounding.

In [0]:
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]

# Load Gold table sorted from coldest to warmest band
gold_df = (
    spark.table(f"{CATALOG}.gold.price_by_temperature_band")
    .orderBy("temperature_band_order")
)

# Format Gold rows with dynamically computed skewness so the LLM never does mental math
gold_rows = gold_df.collect()
gold_summary_text = "\n".join([
    f"- Band '{r['temperature_band']}' (order {r['temperature_band_order']}): "
    f"observation_count={r['observation_count']} hrs, "
    f"avg_price={r['average_price_nok_per_kwh']:.4f} NOK/kWh, "
    f"median_price={r['median_price_nok_per_kwh']:.4f} NOK/kWh "
    f"({'right-skewed: avg > median' if r['average_price_nok_per_kwh'] > r['median_price_nok_per_kwh'] else 'left-skewed/equal: avg <= median'}, "
    f"diff={r['average_price_nok_per_kwh'] - r['median_price_nok_per_kwh']:+.4f}), "
    f"min={r['minimum_price_nok_per_kwh']:.4f}, "
    f"max={r['maximum_price_nok_per_kwh']:.4f}, "
    f"avg_wind={r['average_wind_speed_mps']:.2f} m/s"
    for r in gold_rows
])

# Dynamically rank bands in Python so rankings adapt to any date range or price area
by_price = sorted(gold_rows, key=lambda x: x["average_price_nok_per_kwh"], reverse=True)
by_obs = sorted(gold_rows, key=lambda x: x["observation_count"], reverse=True)
small_bands = [r for r in gold_rows if r["observation_count"] < 100]

top_price_str = ", ".join(
    f"#{i+1} '{r['temperature_band']}' ({r['average_price_nok_per_kwh']:.4f} NOK/kWh, {r['observation_count']} hrs)"
    for i, r in enumerate(by_price[:3])
)
top_obs_str = ", ".join(
    f"#{i+1} '{r['temperature_band']}' ({r['observation_count']} hrs, {r['average_price_nok_per_kwh']:.4f} NOK/kWh)"
    for i, r in enumerate(by_obs[:3])
)
small_obs_str = ", ".join(
    f"'{r['temperature_band']}' ({r['observation_count']} hrs)"
    for r in small_bands
) or "None"

prompt = f"""You are a Nordic power market data analyst. Write a 3-bullet executive summary strictly grounded in the dynamic dataset and computed rankings below:

Dataset (ordered coldest to warmest):
{gold_summary_text}

Computed rankings from dataset (use these exact rankings, do not re-rank):
- Top 3 highest average price bands: {top_price_str}
- Top 3 most populated bands (by observation_count): {top_obs_str}
- Small-sample bands (<100 hours): {small_obs_str}

Formatting rules:
- Output ONLY 3 bullet points starting with '• '. Do NOT include any introductory sentence.
- Bullet 1 (Peak Price Bands & Sample Size): Report where average prices peak using the 'Top 3 highest average price bands' ranking, and note any 'Small-sample bands' that should not be over-interpreted.
- Bullet 2 (Core Volume & Temperature Trend): Report the most populated temperature bands using the 'Top 3 most populated bands' ranking and describe how average prices change across the warmer bands.
- Bullet 3 (Price Skewness & Wind): Contrast bands labeled 'right-skewed: avg > median' against bands labeled 'left-skewed/equal: avg <= median', and note how `avg_wind` varies across the bands."""

# Call serverless LLM via parameterized SQL ai_query()
for endpoint in [
    "databricks-meta-llama-3-3-70b-instruct",
    "system.ai.meta-llama-3-3-70b-instruct",
    "databricks-meta-llama-3-1-70b-instruct",
    "databricks-meta-llama-3-1-8b-instruct",
]:
    try:
        result_df = spark.sql(f"""
            SELECT ai_query(
                '{endpoint}',
                :prompt
            ) AS executive_summary
        """, {"prompt": prompt})
        print(f"Using model endpoint: {endpoint}")
        break
    except Exception:
        continue

# Enrich with metadata and persist to Gold for dashboarding
summary_gold_df = (
    result_df
    .withColumn("price_area", F.lit("NO2"))
    .withColumn("generated_at_utc", F.current_timestamp())
)

summary_gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.gold.executive_summary"
)

display(spark.table(f"{CATALOG}.gold.executive_summary"))